# 02_M2_Li_Paper_Modality_Comparison.ipynb

M2: complete Li-inspired binary feature-selection experiments for current, vibration, and torque.

In [ ]:
from pathlib import Path
import json
import os
import sys

ROOT_CANDIDATES = [Path.cwd(), Path.cwd().parent, Path('/kaggle/working/pinn-motor-fault-code-v2')]
PROJECT_ROOT = next((path.resolve() for path in ROOT_CANDIDATES if (path / 'src' / 'pinn_motor_fault').is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Cannot locate the repository source tree. Set the notebook working directory to the project root.')
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
REPORTS_DIR = PROJECT_ROOT / 'reports'
if os.environ.get('PINN_PADERBORN_DATA'):
    DATA_DIR = Path(os.environ['PINN_PADERBORN_DATA']).expanduser().resolve()
else:
    KAGGLE_DATA_DIR = Path('/kaggle/input/paderborn')
    LOCAL_DATA_DIR = PROJECT_ROOT / 'data' / 'paderborn' / 'extracted'
    DATA_DIR = KAGGLE_DATA_DIR if KAGGLE_DATA_DIR.is_dir() else LOCAL_DATA_DIR
print('Project root:', PROJECT_ROOT)
print('Paderborn data:', DATA_DIR)


## Complete three-modality experiment
Each modality is loaded, split by recording, selected using training recordings only, trained, evaluated, and saved to its own immutable experiment directory. Populations and split manifests are recorded per modality. The results are directly comparable only when both the recording populations and train/test recording assignments match.

In [ ]:
from pinn_motor_fault.paderborn import MODALITY_SAMPLE_RATES_HZ, load_paderborn_windows_with_metadata
from pinn_motor_fault.train import run_paper_feature_experiment

WINDOW_SIZE = 4096
STRIDE = 2048
MAX_WINDOWS_PER_RECORDING = 16
SEED = 17
EPOCHS = 60
BATCH_SIZE = 64
MODALITY_EXPERIMENTS = {
    'current': 'M2_1_current',
    'vibration': 'M2_2_vibration',
    'torque': 'M2_3_torque',
}
assert DATA_DIR.is_dir(), f'Paderborn data directory does not exist: {DATA_DIR}'
comparison = {}
for modality, experiment_id in MODALITY_EXPERIMENTS.items():
    loaded = load_paderborn_windows_with_metadata(
        DATA_DIR, window_size=WINDOW_SIZE, stride=STRIDE,
        signal_preference=modality, max_windows_per_file=MAX_WINDOWS_PER_RECORDING,
        require_modality_match=True,
    )
    assert loaded.modality == modality
    assert loaded.windows.size > 0 and len(loaded.sources) == loaded.labels.size
    assert all(Path(source).suffix.lower() == '.mat' for source in loaded.sources), 'Expected real Paderborn MAT recordings.'
    data_root = DATA_DIR.resolve()
    assert all(Path(source).resolve().is_relative_to(data_root) for source in loaded.sources), 'Loaded recordings must be under DATA_DIR.'
    experiment_dir = REPORTS_DIR / experiment_id
    metrics = run_paper_feature_experiment(
        loaded.windows, loaded.labels, loaded.sources, experiment_dir,
        modality=modality, sample_rate_hz=MODALITY_SAMPLE_RATES_HZ[modality],
        skipped=loaded.skipped, epochs=EPOCHS, batch_size=BATCH_SIZE, seed=SEED,
        dataset_name='real_paderborn',
    )
    split = json.loads((experiment_dir / 'split_manifest.json').read_text(encoding='utf-8'))
    config = json.loads((experiment_dir / 'config.json').read_text(encoding='utf-8'))
    comparison[modality] = {
        'experiment_id': experiment_id,
        'accuracy': metrics['accuracy'],
        'macro_f1': metrics['macro_f1'],
        'balanced_accuracy': metrics['balanced_accuracy'],
        'source_recordings': config['source_recordings'],
        'train_files': split['train_files'],
        'test_files': split['test_files'],
        'frequency_resolution_hz': config['frequency_resolution_hz'],
    }
reference = comparison['current']
for modality, record in comparison.items():
    same_population = set(record['source_recordings']) == set(reference['source_recordings'])
    same_split = record['train_files'] == reference['train_files'] and record['test_files'] == reference['test_files']
    record['same_population_as_current'] = same_population
    record['same_split_as_current'] = same_split
    record['directly_comparable_to_current'] = same_population and same_split
print(json.dumps(comparison, indent=2))
print('Modality runs have separate source populations and split manifests; interpret cross-modality metrics only where directly_comparable_to_current is true.')
